
objective function
$$
\min \sum_{j \in S} c_j x_j + \sum_{i=1}^{10} p_i \big( \sum_{j \in S_i} x_j - 1 \big)
$$

constraint:
$\forall i=1,...k$:
$$
\sum_{j \in S_i} x_j \geq 1 
$$
and $\forall j=1,...k$:
$$
x_j \in {0,1}
$$

In [5]:
# Exercise 3
# Question 11: MILP with Binary, so we get an optimal solution)

using GLPK, JuMP

L = collect(1:10)
# subsets/subroutes
S = [
    [1, 2, 5],      # S1
    [1, 3, 4],      # S2
    [2, 6],         # S3
    [3, 7],         # S4
    [4, 5, 8],      # S5
    [6, 9],         # S6
    [7, 10],        # S7
    [8, 9, 10],     # S8
    [1, 6, 8],      # S9
    [2, 4, 7],      # S10
    [1, 5, 9],      # S11
    [6, 10]         # S12
]
c = [9, 8, 7, 6, 10, 11, 12, 14, 13, 9, 10, 8]  # cost
rho = [2, 1, 1, 3, 1, 2, 2, 1, 3, 2]            # penalty

K = length(S)
N = length(L)

model = Model(GLPK.Optimizer)

@variable(model,x[1:K],Bin)
# @variable(model,1>=x[1:K]>=0) # LP- relaxation constraint is here 

@objective(model, Min, 
    sum( c[route]*x[route] for route=1:K )
    +
    sum( rho[loc] * ( sum( x[route] for route=1:K if loc in S[route] ) - 1) for loc=1:N)
    )

@constraint(model, [loc=1:N], sum( x[route] for route=1:K if loc in S[route]) >= 1)
# @constraint(model, x[3]==0) # Calculating LPrelaxation of the branches for question 12 is here

optimize!(model)

println("solver status: ", termination_status(model))
println("solution status: ", primal_status(model))
println("object value: ", objective_value(model))

print("\n")
for route=1:K
    println(value(x[route]))
end

solver status: OPTIMAL
solution status: FEASIBLE_POINT
object value: 44.0

0.0
0.0
1.0
1.0
1.0
0.0
0.0
0.0
0.0
0.0
1.0
1.0


In [ ]:
# Exercise 3
# Question 13
# Relaxation with only one constraint

using GLPK, JuMP

L = collect(1:10)
S = [
    [1, 2, 5],      # S1
    [1, 3, 4],      # S2
    [2, 6],         # S3
    [3, 7],         # S4
    [4, 5, 8],      # S5
    [6, 9],         # S6
    [7, 10],        # S7
    [8, 9, 10],     # S8
    [1, 6, 8],      # S9
    [2, 4, 7],      # S10
    [1, 5, 9],      # S11
    [6, 10]         # S12
]
c = [9, 8, 7, 6, 10, 11, 12, 14, 13, 9, 10, 8]
rho = [2, 1, 1, 3, 1, 2, 2, 1, 3, 2]

K = length(S)
N = length(L)

model = Model(GLPK.Optimizer)


@variable(model,x[1:K],Bin)
 #@variable(model,1>=x[1:K]>=0)
@objective(model, Min, 
    sum( c[route]*x[route] for route=1:K )
    )

@constraint(model,sum( sum( x[route] for route=1:K if loc in S[route]) for loc=1:N) >= N)

optimize!(model)

println("solver status: ", termination_status(model))
println("solution status: ", primal_status(model))
println("object value: ", objective_value(model))

print("\n")
for route=1:K
    println(value(x[route]))
end

solver status: OPTIMAL
solution status: FEASIBLE_POINT
object value: 30.0

0.0
1.0
1.0
1.0
0.0
0.0
0.0
0.0
0.0
1.0
0.0
0.0


In [ ]:
# Question 14
L = collect(1:10)
S = [
    [1, 2, 5],      # S1
    [1, 3, 4],      # S2
    [2, 6],         # S3
    [3, 7],         # S4
    [4, 5, 8],      # S5
    [6, 9],         # S6
    [7, 10],        # S7
    [8, 9, 10],     # S8
    [1, 6, 8],      # S9
    [2, 4, 7],      # S10
    [1, 5, 9],      # S11
    [6, 10]         # S12
]
c = [9, 8, 7, 6, 10, 11, 12, 14, 13, 9, 10, 8]

K = 1:length(S)
N = 1:length(L)

ratio = [c[k]/length(S[k]) for k=K]

cost = 0
for i=N    
    cost_i = minimum(ratio[k] for k=K if i in S[k])
    cost = cost + cost_i
end
print(cost)